In [3]:
## Imports

import sys, os
from pathlib import Path

parent_folder = str(Path.cwd().parents[0])
if parent_folder not in sys.path:
    sys.path.append(parent_folder)

import scipy
import pickle
import seaborn as sns
import sigpy as sp
import cupy as cp
import numpy as np
from scipy.io import savemat
import twixtools
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from scipy import ndimage
import recon_plot_helpers
import save_data_helpers

### Load ADMM results

In [12]:
import numpy as np
import os

# Base directory
base_dir = "/data/lilianae/ADMM_results/admm_400sp_z_rho_0.01_beta_0.0001_lam1e-3_15iters/iters"

# Iteration to unpack
iter_num = 2
npz_path = os.path.join(base_dir, f"iter_{iter_num:03d}.npz")

# Load npz file
data = np.load(npz_path)

def reshape_mvf(mvf_flat, mvf_shape, num_gates):
    mvf = np.reshape(mvf_flat, (*mvf_shape, 3, num_gates), order='F')
    mvf = np.moveaxis(mvf, -1, 0)
    return mvf

# Inspect available arrays
print("Available arrays in npz:", data.files)

# Unpack
num_gates = 5
img_shape = (58, 512, 512) 
mvf_shape = (58, 512, 512) 
S_inv = reshape_mvf(data["S_inv"], mvf_shape, num_gates)      # shape: (num_gates, z, y, x, 3)
S = reshape_mvf(data["S"], mvf_shape, num_gates)
z_abs = np.abs(data["z"])      # shape: (num_gates, x, y, z)
lam_abs = np.abs(data["lam"])

print(f"S_inv.shape = {S_inv.shape}")
print(f'S.shape = {S.shape}')
print(f"z_abs.shape = {z_abs.shape}")
print(f'lam_abs.shape = {lam_abs.shape}')


Available arrays in npz: ['z', 'lam', 'u', 'S', 'S_inv']
S_inv.shape = (5, 58, 512, 512, 3)
S.shape = (5, 58, 512, 512, 3)
z_abs.shape = (5, 58, 512, 512)
lam_abs.shape = (58, 512, 512)


In [13]:
print(lam_abs.shape)
print(lam_abs.min())
print(lam_abs.max())

(58, 512, 512)
2.8373976e-22
0.98353696


In [16]:
print(z_abs[0].max())

0.99376816


### Motion vector field statistics

In [ ]:
# for gate in range(num_gates):
#     print(f'Gate {gate+1}:')
#     print(" Motion vector statistics:")
#     print(f"    X motion range: {S_inv[gate, :, :, :, 0].min():.3f} to {S_inv[gate, :, :, :, 0].max():.3f}")
#     print(f"    Y motion range: {S_inv[gate, :, :, :, 1].min():.3f} to {S_inv[gate, :, :, :, 1].max():.3f}")
#     print(f"    Z motion range: {S_inv[gate, :, :, :, 2].min():.3f} to {S_inv[gate, :, :, :, 2].max():.3f}")

#     # Check for spatial coherence
#     print(f"    Motion field shape: {S_inv.shape}")

#     # Look at motion magnitude distribution
#     magnitude = np.sqrt(np.sum(S_inv[0]**2, axis=-1))
#     print(f"    Magnitude range: {magnitude.min():.3f} to {magnitude.max():.3f}")

### Save MVF for optical flow comparison

In [ ]:
# # Extract 2nd and 3rd images (indices 1 and 2), slice 256, xy vectors
# slice_2_vectors_inv = S_inv[2, :, 128:384, 256, :2]  # (58, 512, 2) - x,y components
# slice_3_vectors_inv = S_inv[3, :, 128:384, 256, :2]  # (58, 512, 2)

# slice_2_vectors = S[2, :, 128:384, 256, :2]
# slice_3_vectors = S[3, :, 128:384, 256, :2]

# # OR if S_inv already contains the motion vectors (not positions):
# u_python_inv = slice_3_vectors_inv[:, :, 0]  # Use whichever frame you want
# v_python_inv = slice_3_vectors_inv[:, :, 1]

# u_python = slice_3_vectors[:, :, 0]  # Use whichever frame you want
# v_python = slice_3_vectors[:, :, 1]

# print(f"u_python_inv.shape = {u_python_inv.shape}")  # Should be (58, 512)
# print(f"v_python_inv.shape = {v_python_inv.shape}")  # Should be (58, 512)

# print(f"u_python.shape = {u_python.shape}")  # Should be (58, 512)
# print(f"v_python.shape = {v_python.shape}")  # Should be (58, 512)

# # Also save the image slice for reference
# image_2 = z_abs[2, :, 128:384, 256]  # Assuming this is image data

# # Save to .mat file for MATLAB
# savemat('python_vectors_cropped.mat', {
#     'u_python' : u_python,
#     'v_python' : v_python,
#     'u_python_inv': u_python_inv,
#     'v_python_inv': v_python_inv,
#     'image_2': image_2
# })

# print("Saved python_vectors_cropped.mat")

### View MVFs as quiver plot

In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt
# from mpl_toolkits.mplot3d import Axes3D

# def plot_3d_quiver(vector_field, gate, spacing=5, scale=1.0, alpha=0.6, 
#                    color='blue', figsize=(12, 10), title="3D Vector Field"):
#     """
#     Create a 3D quiver plot for a 3D vector field.
    
#     Parameters:
#     - vector_field: 3D array of shape (x, y, z, 3) where the last dimension contains [u, v, w] components
#     - spacing: spacing between vectors (higher = fewer vectors, cleaner plot)
#     - scale: scaling factor for vector lengths
#     - alpha: transparency of vectors
#     - color: color of vectors
#     - figsize: figure size
#     - title: plot title
    
#     Returns:
#     - fig, ax: matplotlib figure and axes objects
#     """
    
#     ## Get dimensions
    
#     vector_field_gate = vector_field[gate]
#     nz, ny, nx, _ = vector_field_gate.shape
    
#     ## Create coordinate grids with specified spacing
#     x = np.arange(0, nx, spacing)
#     y = np.arange(0, ny, spacing)
#     z = np.arange(0, nz, spacing)
#     Z, Y, X = np.meshgrid(z, y, x, indexing='ij')
    
#     ## Extract vector components at grid points
#     U = vector_field_gate[::spacing, ::spacing, ::spacing, 0]
#     V = vector_field_gate[::spacing, ::spacing, ::spacing, 1]
#     W = vector_field_gate[::spacing, ::spacing, ::spacing, 2]
    
#     ## Create 3D plot
#     fig = plt.figure(figsize=figsize)
#     ax = fig.add_subplot(111, projection='3d')
    
#     ## Create quiver plot
#     ax.quiver(X, Y, Z, U, V, W, 
#               length=scale, normalize=True, alpha=alpha, color=color)
    
#     ## Set labels and title
#     ax.set_xlabel('X')
#     ax.set_ylabel('Y')
#     ax.set_zlabel('Z')
#     ax.set_title(title + f": Gate {gate+1}")
    
#     plt.show()
#     return fig, ax


# def plot_3d_quiver_all_gates(vector_field, spacing=5, scale=1.0, alpha=0.6, 
#                    color='blue', figsize=(20, 12), title="3D Vector Field - All Gates"):
#     """
#     Create a 3D quiver plot for a 3D vector field.
    
#     Parameters:
#     - vector_field: 3D array of shape (x, y, z, 3) where the last dimension contains [u, v, w] components
#     - spacing: spacing between vectors (higher = fewer vectors, cleaner plot)
#     - scale: scaling factor for vector lengths
#     - alpha: transparency of vectors
#     - color: color of vectors
#     - figsize: figure size
#     - title: plot title
    
#     Returns:
#     - fig, ax: matplotlib figure and axes objects
#     """
    
#     # Get dimensions
    
#     num_gates = vector_field.shape[0]
#     colors = ['blue', 'red', 'green', 'orange', 'purple']  # Different color for each gate
    
#     # Create subplots - 1 row, 5 columns
#     fig = plt.figure(figsize=figsize)
#     axes = []
    

#     for gate in range(num_gates):
#         ## Create subplot
#         ax = fig.add_subplot(1, 5, gate+1, projection='3d')
#         axes.append(ax)

#         # Get vector field for that gate
#         vector_field_gate = vector_field[gate]
#         nz, ny, nx, _ = vector_field_gate.shape

#         x = np.arange(0, nx, spacing)
#         y = np.arange(0, ny, spacing)
#         z = np.arange(0, nz, spacing)
#         Z, Y, X = np.meshgrid(z, y, x, indexing='ij')
        
#         # Get vector components at grid points
#         U = vector_field_gate[::spacing, ::spacing, ::spacing, 0]
#         V = vector_field_gate[::spacing, ::spacing, ::spacing, 1]
#         W = vector_field_gate[::spacing, ::spacing, ::spacing, 2]
        
#         # Create 3D plot
#         # fig = plt.figure(figsize=figsize)
#         # ax = fig.add_subplot(111, projection='3d')
        
#         # Create quiver plot
#         ax.quiver(X, Y, Z,  U, V, W,
#                 length=scale, normalize=True, alpha=alpha, color=color)
        
#         # Set labels and title
#         ax.set_xlabel('X')
#         ax.set_ylabel('Y')
#         ax.set_zlabel('Z')
#         ax.set_title(f'Gate {gate+1}')
    
#     fig.suptitle(title, fontsize=16, y =0.75)
#     plt.tight_layout
#     plt.show()
#     return fig, ax




# # Basic 3D quiver plot
# ## Single gate
# fig, ax = plot_3d_quiver(S_inv, gate=1, spacing=8, scale=0.5)

# ## All gates
# fig, ax = plot_3d_quiver_all_gates(S_inv, spacing=8, scale=0.5)